# Notebook 5 — Full Transfer Matrix: Does Truthfulness Generalise?

## The central question

The RepE paper makes a bold claim:

> *Language models encode a **universal** truth direction in their hidden states — one that is
> shared across domains, not specific to any particular dataset or topic.*

If this is true, a probe trained on geography questions should detect lies in arithmetic questions,
and vice versa. Notebook 3 tested one slice of this (training always on `repeng_truthful`).
Here we test the **full NxN transfer matrix**: every dataset as a source, every dataset as a target.

## What the matrix looks like

```
              eval →  cities  larger_than  qa  repeng_truthful
train ↓
cities               [ 0.90 ]  [ ??? ]  [ ??? ]  [ ??? ]
larger_than          [ ??? ]  [ 0.90 ]  [ ??? ]  [ ??? ]
qa                   [ ??? ]  [ ??? ]  [ 0.90 ]  [ ??? ]
repeng_truthful      [ ??? ]  [ ??? ]  [ ??? ]  [ 0.90 ]
```

- **Diagonal** — in-distribution accuracy (train == eval).  
  High diagonal = the probe works at all.  
- **Off-diagonal** — cross-domain transfer accuracy.  
  High off-diagonal = the truthfulness direction is shared across domains.

## RepE paper findings (on larger models)

With Llama-2 and similar large instruction-tuned models, the RepE paper reports:
- Diagonal values near 0.85–0.95
- Off-diagonal values of 0.60–0.80 for semantically related datasets
- Weaker transfer (0.50–0.60) across very different reasoning types

With `distilgpt2` (82M parameters, no instruction tuning), we expect much weaker transfer —
but the *pattern* of which transfers work and which do not is still informative.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import io
from contextlib import redirect_stderr, redirect_stdout
import pandas as pd

from lie_detector_llm.datasets import build_dataset_collection
from lie_detector_llm.experiment import run_full_transfer_matrix
from lie_detector_llm.plotting import plot_transfer_heatmap

collection = build_dataset_collection(include_repeng_truthful=True)
print("Datasets:", collection.dataset_names())

## LR probe — full transfer matrix

In [ ]:
with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    matrix_lr = run_full_transfer_matrix(
        collection=collection,
        model_name='distilgpt2',
        probe_method='lr',
        layer_index=-1,
    )

pivot_lr = matrix_lr.results.pivot(
    index='train_dataset', columns='eval_dataset', values='grouped_accuracy'
).round(2)
pivot_lr

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plot_transfer_heatmap(
    matrix_lr.results,
    title='Full transfer matrix — LR probe (distilgpt2, last layer)',
)
plt.show()

## DIM probe — full transfer matrix

DIM is parameter-free and may generalise differently from LR. Let's compare.

In [ ]:
with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    matrix_dim = run_full_transfer_matrix(
        collection=collection,
        model_name='distilgpt2',
        probe_method='dim',
        layer_index=-1,
    )

pivot_dim = matrix_dim.results.pivot(
    index='train_dataset', columns='eval_dataset', values='grouped_accuracy'
).round(2)

fig, ax = plot_transfer_heatmap(
    matrix_dim.results,
    title='Full transfer matrix — DIM probe (distilgpt2, last layer)',
)
plt.show()
pivot_dim

## Side-by-side comparison: transfer gap

In [ ]:
# Difference matrix: LR − DIM (positive = LR better, negative = DIM better)
diff = (pivot_lr - pivot_dim).round(2)
print("LR − DIM transfer difference (positive = LR better):")
print(diff.to_string())

## All-probe aggregate: average off-diagonal transfer

In [ ]:
import numpy as np

summary_rows = []
for method in ['dim', 'lat', 'lr', 'pca-g']:
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        m = run_full_transfer_matrix(
            collection=collection,
            model_name='distilgpt2',
            probe_method=method,
            layer_index=-1,
        )
    pivot = m.results.pivot(
        index='train_dataset', columns='eval_dataset', values='grouped_accuracy'
    )
    n = len(pivot)
    diagonal_mean = np.diag(pivot.values).mean()
    off_diag = pivot.values[~np.eye(n, dtype=bool)]
    off_diag_mean = off_diag.mean()
    summary_rows.append({
        'probe_method': method,
        'in_distribution_mean': round(diagonal_mean, 3),
        'transfer_mean': round(off_diag_mean, 3),
        'transfer_gap': round(diagonal_mean - off_diag_mean, 3),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('transfer_mean', ascending=False)
summary_df

## Interpretation

### Reading the matrix

| Pattern | What it means |
|---------|---------------|
| High diagonal, high off-diagonal | Strong universal truth direction — the RepE claim holds |
| High diagonal, low off-diagonal | Dataset-specific features drive the probe, not universal truth |
| Low diagonal | The probe is not even working in-distribution — model is too small or prompts are wrong |

### Why `distilgpt2` shows weak transfer

1. **Size** — distilgpt2 has only 82M parameters and 6 layers. The truthfulness
   signal is weaker and less consistent than in GPT-3/Llama-class models.

2. **No instruction tuning** — distilgpt2 was trained purely on next-token prediction.
   It has no explicit incentive to develop a coherent internal model of truth.

3. **Small datasets** — our datasets have only 8–40 groups each. With so few training
   examples, even LR can overfit to surface features of the training domain.

### What the RepE paper claims (and where it holds)

The paper uses **Llama-2-7B-chat** and similar instruction-tuned models with billions of
parameters. Those models show genuinely high off-diagonal transfer (0.65–0.80), supporting
the universal truth-direction hypothesis.

The **pattern** observed here with distilgpt2 — that some probe/dataset combinations
transfer and others do not — is still informative: it shows the methodology works, and
that model scale is a critical factor for strong generalisation.

### Practical conclusion

The `transfer_gap` column in the summary table above is the most important single number:
it measures how much accuracy is lost when moving from in-distribution to cross-domain.
A smaller gap = a more universal, domain-agnostic truth representation.